In [1]:
import pandas as pd
from collections import defaultdict

### Load configuration files

In [2]:
D_prev = pd.read_excel("previous.xlsx").shape[1]-1
X_prev = pd.read_excel("previous.xlsx",index_col=0)
X_curr = pd.read_excel("planned.xlsx", index_col=0)
plan = pd.concat([X_prev, X_curr], axis=1)
R_min = pd.read_excel("demand_min.xlsx",index_col=0)
R_max = pd.read_excel("demand_max.xlsx",index_col=0)
wishes = pd.read_excel("wishes.xlsx",index_col=0)
rules = pd.read_excel("rules.xlsx")
rules.set_index(rules.columns[0], inplace=True)
rules = rules.iloc[:,0]
t = pd.read_excel("working_hours.xlsx",index_col=0)
H = pd.read_excel("shifts.xlsx")
H.set_index(H.columns[0], inplace=True)
H = H.iloc[:,0]
preferences = pd.read_excel("preferences.xlsx",index_col=0)

## Sets and Indices

In [3]:
I = list(pd.read_excel("preferences.xlsx", usecols=[0])["Name"])
I_rotate = []
for i in I:
    if t.loc[i,"percent"] >= 0.6 and i!="HN":
        I_rotate.append(i)
I_nights = [i for i in I if preferences.loc[i,"nights"]==1]

In [4]:
S = list(pd.read_excel("shifts.xlsx", usecols=[0])["Shift"])

In [5]:
D = X_curr.shape[1]
D_e = [d for d in range(D) if (d-rules.loc["first_monday"])%7>=5]
D_s = [d for d in D_e[::2] if d + 1 < D]

D_e_A = []
D_e_B = []

for i in range(0, len(D_e), 2):
    chunk = D_e[i:i+2]
    if (i // 2) % 2 == rules.loc["first_weekend"]:
        D_e_A.extend(chunk)
    else:
        D_e_B.extend(chunk)

## Parameters

In [6]:
# Compute absence days on weekdays which count as 7,7 hours of work
paid_vacation = defaultdict(int)
for i in I:
    for d in range(D):
        if X_curr.loc[i,d] == "U" and not d in D_e:
            paid_vacation[i] += 1

In [7]:
# Compute absence days on weekdays which count as 8 hours of work
paid_offunit = defaultdict(int)
for i in I:
    for d in range(D):
        if X_curr.loc[i,d] == "Z":
            paid_offunit[i] += 1

In [8]:
is_hours = {}
min_nights = {}
all_nights = sum(R_min.loc["N",d] for d in range(D))

for i in I_nights:
    is_hours[i] = t.loc[i,"hours"] * D / 7 - paid_vacation[i]*7.7 - paid_offunit[i]*8.

all_hours = sum(is_hours.values())

for i in I_nights:
    min_nights[i] = int((is_hours[i]/all_hours)*all_nights)

In [9]:
def get_preference_score(i:list,s:list,d:int):
    if pd.isna(wishes.loc[i,d]):
        return preferences.loc[i,s]
    else:
        if wishes.loc[i,d] == "X":
            return -10
        elif wishes.loc[i,d] == s:
            return 5
        else:
            return -5

In [10]:
plan_dict = plan.stack().to_dict()
r_min_dict = R_min.stack().to_dict()
r_max_dict = R_max.stack().to_dict()
h_dict = H.to_dict()
target_hours_dict = t["hours"].to_dict()
pref_max_s_dict = preferences.to_dict('index')
pref_weekend = preferences["weekend"].to_dict()

## ILP Model

In [11]:
import gurobipy as gp
from gurobipy import GRB

schedule = gp.Model("schedule")

Set parameter Username
Set parameter LicenseID to value 2793684
Academic license - for non-commercial use only - expires 2027-03-17


### Decision Variables

In [12]:
x = schedule.addVars(I,S,range(-D_prev,D),vtype=GRB.BINARY,name="x")
cons = schedule.addVars(I, S, range(-1,D), lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="cons")
L_to_E = schedule.addVars(I,range(D), lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="L_to_E")
n_beg = schedule.addVars(I, range(-rules.loc["min_betw_n"], D), vtype=GRB.CONTINUOUS, lb=0.0, ub=1.0, name="n_beg")
n_end = schedule.addVars(I, range(-rules.loc["min_betw_n"], D), vtype=GRB.CONTINUOUS, lb=0.0, ub=1.0, name="n_end")
hour = schedule.addVars(I, vtype=GRB.CONTINUOUS, lb=-GRB.INFINITY, name="hour")
hour_max = schedule.addVar(vtype=GRB.CONTINUOUS, lb=-GRB.INFINITY, name="hour_max")

TypeError: 'numpy.float64' object cannot be interpreted as an integer

### Objective Function

In [13]:
# 2
schedule.setObjectiveN(hour_max, index=0, priority=3, abstol=5, name="Min_HourMax")

# 3
pref_dict = {(i, s, d): get_preference_score(i, s, d) for i in I for s in S for d in range(D)}
schedule.setObjectiveN(-x.prod(pref_dict), index=2, priority=2, name="Max_Preference")

# 4
schedule.setObjectiveN(L_to_E.sum(), index=3, priority=1, name="Min_L_to_E")

# 5
schedule.setObjectiveN(-cons.sum(), index=1, priority=0, name="Max_Cons")

schedule.ModelSense = GRB.MINIMIZE

### Constraints

#### Basic Scheduling and Constants

In [14]:
# 7a
schedule.addConstrs((x.sum(i, '*', d) <= 1 for i in I for d in range(-D_prev, D)));

In [15]:
# 7b
for (i, d), val in plan_dict.items():
    if val in ["X", "Z", "U"]:
        for s in S:
            x[i, s, d].ub = 0.0

In [16]:
# 7c
for (i, d), val in plan_dict.items():
    if val in S:
        x[i, val, d].lb = 1.0

#### Staffing Requirements and Load Balancing

In [17]:
# 8a
schedule.addConstrs((x.sum('*', s, d) >= r_min_dict[s, d] for s in S for d in range(D)));

In [18]:
# 8b
schedule.addConstrs((x.sum('*', s, d) <= r_max_dict[s, d] for s in S for d in range(D)));

In [19]:
# 8c
for i in I:
    vac_hours = paid_vacation[i] * 7.7
    off_hours = paid_offunit[i] * 8.0
    work_hours = target_hours_dict[i] * D / 7.0
    rhs_constant = work_hours - vac_hours - off_hours
    schedule.addConstr(gp.quicksum(h_dict[s] * x[i, s, d] for s in S for d in range(D)) - hour[i] == rhs_constant)

In [20]:
# 8d
schedule.addConstrs(hour_max >= hour[i] for i in I);

#### Consecutive Shifts and Tracking Variables

In [21]:
# 9a
for i in I:
    max_k = pref_max_s_dict[i].get("max_cons_s", 0)
    window_size = max_k + 1
    for d in range(-max_k, D - max_k):
        window = range(d, d + window_size)
        fixed_work = (plan.loc[i, window] == "Z").sum()
        schedule.addConstr(x.sum(i, '*', window) + fixed_work <= max_k)

In [22]:
# 9b
for i in I:
    for s in S:
        col_name = f"max_cons_{s}"
        max_k = pref_max_s_dict[i].get(col_name, 0)
        if max_k > 0:
            max_k = int(max_k)
            window_size = max_k + 1
            for d in range(-max_k, D - max_k):
                window = range(d, d + window_size)
                schedule.addConstr(x.sum(i, s, window) <= max_k)

In [23]:
# 9c
schedule.addConstrs((cons[i,s,d] <= x[i,s,d] for i in I for s in S for d in range(-1,D-1)));

In [24]:
# 9d
schedule.addConstrs((cons[i,s,d] <= x[i,s,d+1] for i in I for s in S for d in range(-1,D-1)));

In [25]:
# 9e
schedule.addConstrs(L_to_E[i, d] >= x[i, "L", d-1] + x[i, "E", d] - 1 for i in I for d in range(D));

#### Night Shift Block Regulations

In [26]:
# 10a
schedule.addConstrs(x.sum(i, "N", range(D)) >= min_nights[i] for i in I_nights);

In [27]:
# 10b
schedule.addConstrs((n_beg[i,d] >= x[i,"N",d] - x[i,"N",d-1] for i in I for d in range(D)));

In [28]:
# 10c
schedule.addConstrs((n_beg[i,d] <= x[i,"N",d] for i in I for d in range(D)))
schedule.addConstrs((n_beg[i,d] <= 1 - x[i,"N",d-1] for i in I for d in range(D)));

In [29]:
# 10d
schedule.addConstrs((n_end[i,d] >= x[i,"N",d] - x[i,"N",d+1] for i in I for d in range(-rules.loc["min_betw_n"], D-1)));

In [30]:
# 10e
schedule.addConstrs((n_end[i,d] <= x[i,"N",d] for i in I for d in range(-rules.loc["min_betw_n"], D-1)));

In [31]:
# 10f
min_post = int(rules.loc["min_post_n"])

for i in I:
    for d in range(-min_post, D):
        for j in range(d+1,min(D,d+min_post+1)):
            fixed_work = 1 if plan.loc[i,j] == "Z" else 0
            schedule.addConstr(n_end[i, d] <= 1 - x.sum(i, S,j) - fixed_work)

In [32]:
# 10g
schedule.addConstrs(n_beg[i,d]+n_end[i,d]<=1 for i in I for d in range(D));

In [33]:
# 10h
for i in I:
    for d in range(1,D):
        if plan.loc[i,d]== "U" and plan.loc[i, d - 1]!= "U":
            x[i,"N",d-1].ub = 0.0
            x[i,"L",d-1].ub = 0.0

In [34]:
# 10i
min_b = int(rules.loc["min_betw_n"])
for i in I:
    for d in range(-min_b, D):
        forbidden_starts = range(d + 1, min(d + min_b + 1, D))
        if forbidden_starts:
            schedule.addConstr(n_end[i, d] + n_beg.sum(i, forbidden_starts) <= 1)

In [35]:
# 10j
max_night_blocks = int(rules.loc["max_per_n"])
schedule.addConstrs(n_beg.sum(i, range(D)) + n_end.sum(i, range(D)) <= 2*max_night_blocks for i in I);

#### Weekend and Rotation Rules

In [36]:
# 11a
schedule.addConstrs((x[i, s, d] == x[i, s, d+1] for s in S for i in I for d in D_s));

In [37]:
# 11b
schedule.addConstrs(x.sum(i, "N", D_e) <= rules.loc["V_n"] for i in I);

In [38]:
# 11c - 11g
for i in I:
    assignment = pref_weekend.get(i)
    if pd.notna(assignment):
        if assignment == 0:
            work_days, off_days = D_e_A, D_e_B
        else:
            work_days, off_days = D_e_B, D_e_A

        # 11c + 11d
        for d in off_days:
            for s in S:
                x[i, s, d].ub = 0.0

        # 11e + 11f
        for d in work_days:
            if plan.loc[i,d] not in ["X", "Z", "U"]:
                schedule.addConstr(x.sum(i, '*', d) == 1)
    else:
        # 11g
        schedule.addConstr(x.sum(i, '*', D_e) <= D // 7)

In [39]:
# 11h
for i in I_rotate:
    for s in S:
        for d in range(-D_prev, D - rules.loc["W_r"] + 1):
            window = range(d, d + rules.loc["W_r"])
            if rules.loc["A_r"] >= (plan.loc[i, window] == "Z").sum() + (plan.loc[i, window] == "U").sum():
                schedule.addConstr(x.sum(i, s, window) >= 1)

## Solve

In [40]:
schedule.optimize()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[x86] - Darwin 24.6.0 24G720)

CPU model: Intel(R) Core(TM) i5-8257U CPU @ 1.40GHz
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 14732 rows, 7741 columns and 102364 nonzeros (Min)
Model fingerprint: 0x0594accd
Model has 1 linear objective coefficients
Variable types: 4081 continuous, 3660 integer (3660 binary)
Coefficient statistics:
  Matrix range     [1e+00, 9e+00]
  Objective range  [1e+00, 1e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+02]


---------------------------------------------------------------------------
Multi-objectives: starting optimization (min) with 4 objectives... 
---------------------------------------------------------------------------

Multi-objectives: applying initial presolve...
---------------------------------------------------------------------------

Presolve removed 9716 rows and 4122 columns
Presolve time: 0.15s
Presolve

In [41]:
if schedule.status == GRB.OPTIMAL:
    print("Optimal solution found")
elif schedule.status == GRB.INFEASIBLE:
    print("Model is infeasible")
elif schedule.status == GRB.UNBOUNDED:
    print("Model is unbounded")
else:
    print("Status:", schedule.status)

Optimal solution found


## Output solution

In [42]:
schedule_out = pd.DataFrame("X", index=I, columns=range(D))
wishes_out = pd.DataFrame("X", index=I, columns=range(D))

for i in I:
    for d in range(D):
        for s in S:
            if x[i, s, d].Xn > 0.5:
                schedule_out.loc[i, d] = s
                break

for i in I:
    for d in range(D):
        if pd.notna(wishes.loc[i,d]):
            wishes_out.loc[i,d] = "Y" if schedule_out.loc[i,d] == wishes.loc[i,d] else "N"

for i in I:
    for d in range(D):
        if schedule_out.loc[i, d] == "X"  and plan.loc[i,d] == "U":
            schedule_out.loc[i, d] = "U"
        if schedule_out.loc[i, d] == "X" and plan.loc[i,d] == "Z":
            schedule_out.loc[i, d] = "Z"

schedule_out["overtime"] = 0
for i in I:
    schedule_out.loc[i,"overtime"] = hour[i].Xn

/var/folders/48/jswtxhvj005c_k5ggvk4lkz80000gn/T/ipykernel_2086/564289387.py:25: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3.1950000000000003' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  schedule_out.loc[i,"overtime"] = hour[i].Xn


In [43]:
with pd.ExcelWriter(
    "out.xlsx",
    engine="openpyxl",
    mode="a",
    if_sheet_exists="overlay"
) as writer:
    schedule_out.to_excel(writer, sheet_name="Schedule", index=True)

In [44]:
with pd.ExcelWriter(
    "wishes_out.xlsx",
    engine="openpyxl",
    mode="a",
    if_sheet_exists="overlay"
) as writer:
    wishes_out.to_excel(writer, sheet_name="Wishes", index=True)